# ViT-Motion v3 — a real sky mask, and the interpretability arc re-run on the new collection

What changed since v2:

1. **The mask.** v2 defined "sky" as the top half of the frame. Audited against the aligned
   depth of the 2026-08-19 collection, that region is only ~29% sky — the rest is distant
   structure (~48%), mid-range terrain (~18%) and even a little near ground (~3%). The sky's
   real area share is **0.146 ± 0.089**, not 0.50, so every "saliency-to-area ratio" in v2 was
   normalized by a constant that is off by ~3.4x.
2. **The metric.** Saliency is now reported per region as `fraction / area`
   (*concentration*): 1.0 means the model attends to a region exactly in proportion to its
   size. This is scale-free, so frames with a lot of sky and frames with almost none are
   comparable — the v2 fraction was not.
3. **The intervention.** The RRR penalty now acts on the per-frame mask, with a
   `nonground` variant that penalizes *everything beyond usable range* (sky **and** distant
   structure), which is what the depth audit says is uninformative for egomotion.
4. **The test.** The 2026-08-19 collection (2 wet + 3 dry runs, a different day and site)
   is evaluated as a held-out condition, with the *training* normalization, so the numbers
   line up with every earlier result.

**Attach:** the code dataset, the v0.2.1 training dataset, the artifacts dataset (checkpoint),
and the new-data dataset (`processed/` with `rgb/`, `depth/`, `samples.csv`, `samples_full.csv`).
**Settings:** GPU on, Internet on (the segmentation weights are downloaded once).

In [ ]:
# --- 1) stage code, install deps, locate every input -------------------------------------
import os, sys, glob, json, shutil, pathlib, subprocess

print('Attached inputs:')
for d in sorted(glob.glob('/kaggle/input/*')):
    print('   ', d)

codes = glob.glob('/kaggle/input/**/precompute_sky_masks.py', recursive=True) \
     or glob.glob('/kaggle/input/**/inspect_dataset.py', recursive=True)
assert codes, 'CODE dataset not attached (looking for precompute_sky_masks.py / inspect_dataset.py).'
CODE_SRC = str(pathlib.Path(sorted(codes, key=len)[0]).parent)

WORK = '/kaggle/working/vit_motion_project'
if os.path.exists(WORK):
    shutil.rmtree(WORK)
shutil.copytree(CODE_SRC, WORK)
os.chdir(WORK); sys.path.insert(0, WORK)
print('code ->', WORK)

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'timm>=1.0', 'transformers>=4.40', 'opencv-python-headless>=4.9'], check=False)

def find_root(marker, exclude=()):
    hits = [pathlib.Path(p).parent.parent for p in
            glob.glob(f'/kaggle/input/**/{marker}', recursive=True)]
    hits = [h for h in hits if not any(x in str(h) for x in exclude)]
    return sorted({str(h) for h in hits}, key=len)

train_roots = [r for r in find_root('samples.csv') if 'new' not in r.lower()]
new_roots   = [r for r in find_root('samples_full.csv')]
ckpts = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True), key=len)

DATA_ROOT = train_roots[0] if train_roots else None
NEW_ROOT  = new_roots[0] if new_roots else None
CKPT      = ckpts[0] if ckpts else None
print('DATA_ROOT', DATA_ROOT)
print('NEW_ROOT ', NEW_ROOT)
print('CKPT     ', CKPT)
assert DATA_ROOT and CKPT, 'training data and checkpoint are required'

In [ ]:
# --- 2) manifest for the TRAINING collection (the model's own splits) --------------------
CFG = 'config_run.yaml'
import yaml
cfg = yaml.safe_load(open('config_kaggle.yaml'))
cfg['data']['root'] = DATA_ROOT
cfg['model']['pretrained'] = False          # the checkpoint already carries encoder weights
yaml.safe_dump(cfg, open(CFG, 'w'))

if not os.path.exists('artifacts/manifest/manifest.csv'):
    !python inspect_dataset.py --config {CFG} --data-root "{DATA_ROOT}"
import pandas as pd
mf = pd.read_csv('artifacts/manifest/manifest.csv')
print(mf['split'].value_counts().to_dict(), '·', mf['experiment_id'].nunique(), 'experiments')

## Step 1 — build the per-frame region masks

One SegFormer pass per unique frame, cached as an 8-bit region-code PNG at the model's input
geometry. `--masker auto` uses the segmentation model when its weights can be fetched and
falls back to the download-free energy-optimization detector (Shen & Wang 2013) otherwise —
so this cell cannot silently leave the pipeline running without masks.

In [ ]:
!python precompute_sky_masks.py --manifest artifacts/manifest/manifest.csv \
    --output-dir artifacts/sky_masks --masker auto --size 224 224 --batch-size 16 --skip-existing
print(open('artifacts/sky_masks/index.json').read()[:1200])

## Step 2 — audit the mask against depth (the falsifiable check)

Depth cannot separate sky from a distant tower — both return nothing — so it answers a
different question: which pixels are *beyond usable range*. Real sky must be a subset of
that region, which makes containment a genuine test. This also re-measures how wrong the
v2 top-half heuristic was, on this data.

In [ ]:
if NEW_ROOT:
    !python validate_sky_mask.py --data-root "{NEW_ROOT}" \
        --output-dir artifacts/mask_validation --num-per-exp 40 --masker auto --qualitative 6
    from IPython.display import Image, display
    display(Image('artifacts/mask_validation/f_mask_qualitative.png'))
    display(Image('artifacts/mask_validation/f_mask_audit.png'))
else:
    print('new-data dataset not attached — skipping the depth audit')

In [ ]:
# Cross-check the two RGB maskers against each other on the same frames.
if NEW_ROOT:
    !python validate_sky_mask.py --data-root "{NEW_ROOT}" \
        --output-dir artifacts/mask_validation_energy --num-per-exp 40 --masker energy --qualitative 3
    a = json.load(open('artifacts/mask_validation/mask_validation.json'))
    b = json.load(open('artifacts/mask_validation_energy/mask_validation.json'))
    for name, d in (('segmentation', a), ('energy', b)):
        print(f"{name:13s} masker={d['masker']:22s} "
              f"sky area {d['frame_composition']['sky_area']['mean']:.3f} "
              f"containment {d['mask_validation']['sky_containment_in_far']['mean']:.3f}")

## Step 3 — quantify the baseline, region by region

`concentration = fraction / area`. The legacy top-half number is printed alongside so this
run can be lined up against the v2 slide.

In [ ]:
!python interpret_quantify.py --config {CFG} --checkpoint "{CKPT}" \
    --sky-mask-dir artifacts/sky_masks --experiments auto --max-exp 6 --num-per-exp 40 \
    --target yaw --saliency inputgrad --tag baseline
from IPython.display import Image, display
display(Image('artifacts/quantify/regions_baseline.png'))
base = json.load(open('artifacts/quantify/regions_baseline.json'))
for region, block in base['regions'].items():
    if block['concentration']['mean'] is not None:
        print(f"{region:7s} fraction {block['fraction']['mean']:.3f} "
              f"area {block['area']['mean']:.3f} -> {block['concentration']['mean']:.2f}x")
print('legacy top-half fraction', base['legacy_top_half']['fraction']['mean'])

## Step 4 — the intervention, three ways

* `sky` (λ=1) — the direct successor to v2, now on a real mask.
* `nonground` (λ=1) — penalize everything not ground, i.e. sky *and* distant structure.
* `sky` (λ=0) — the control: identical extra training, no penalty. If accuracy moves here
  too, the effect is not the penalty.

In [ ]:
RUNS = {
    'rrr_sky':       ['--mask-mode', 'sky',       '--lambda-rrr', '1.0'],
    'rrr_nonground': ['--mask-mode', 'nonground', '--lambda-rrr', '1.0'],
    'control_l0':    ['--mask-mode', 'sky',       '--lambda-rrr', '0.0'],
}
for name, extra in RUNS.items():
    out = f'artifacts/runs/{name}/best.pt'
    cmd = ['python', 'finetune_rrr.py', '--config', CFG, '--checkpoint', CKPT,
           '--sky-mask-dir', 'artifacts/sky_masks', '--epochs', '3', '--target', 'yaw',
           '--max-steps', '200', '--output', out] + extra
    print('\n===', name, '===')
    subprocess.run(cmd, check=True)

In [ ]:
# quantify every fine-tuned model with the same mask and the same metric
for name in RUNS:
    !python interpret_quantify.py --config {CFG} --checkpoint artifacts/runs/{name}/best.pt \
        --sky-mask-dir artifacts/sky_masks --experiments auto --max-exp 6 --num-per-exp 40 \
        --target yaw --saliency inputgrad --tag {name}

rows = []
for tag in ['baseline'] + list(RUNS):
    d = json.load(open(f'artifacts/quantify/regions_{tag}.json'))
    rows.append({'run': tag,
                 'sky_conc': d['regions']['sky']['concentration']['mean'],
                 'ground_conc': d['regions']['ground']['concentration']['mean'],
                 'other_conc': d['regions']['other']['concentration']['mean'],
                 'legacy_half_fraction': d['legacy_top_half']['fraction']['mean']})
import pandas as pd; display(pd.DataFrame(rows).round(3))

## Step 5 — before / after maps, drawn on the mask

In [ ]:
EXP = sorted(mf.loc[mf['split'] == 'test', 'experiment_id'].astype(str).unique())[0]
!python compare_interpret.py --config {CFG} \
    --checkpoint-baseline "{CKPT}" --checkpoint-finetuned artifacts/runs/rrr_sky/best.pt \
    --sky-mask-dir artifacts/sky_masks --experiment "{EXP}" --num-samples 4 \
    --target yaw --saliency inputgrad --output artifacts/interpretability/before_after_v3.png
display(Image('artifacts/interpretability/before_after_v3.png'))

## Step 6 — the held-out collection

A different day, a different site, and two of the five runs in the wet. The manifest is
built with `--split-override test` (so it can never leak into training) and
`--normalization-from` the trained model's own statistics (so the numbers stay comparable).

In [ ]:
if NEW_ROOT:
    cfg_new = yaml.safe_load(open('config_newdata.yaml'))
    cfg_new['data']['root'] = NEW_ROOT
    yaml.safe_dump(cfg_new, open('config_newdata_run.yaml', 'w'))
    !python inspect_dataset.py --config config_newdata_run.yaml --data-root "{NEW_ROOT}" \
        --split-override test --normalization-from artifacts/manifest/normalization.json
    !python precompute_sky_masks.py --manifest artifacts/manifest_newdata/manifest.csv \
        --output-dir artifacts/sky_masks_newdata --masker auto --size 224 224 --skip-existing

In [ ]:
if NEW_ROOT:
    pairs = [f'baseline={CKPT}'] + [f'{n}=artifacts/runs/{n}/best.pt' for n in RUNS]
    !python evaluate_newdata.py --config config_newdata_run.yaml \
        --checkpoints {' '.join(pairs)} --output-dir artifacts/newdata_eval --batch-size 32
    display(Image('artifacts/newdata_eval/f_newdata_accuracy.png'))

In [ ]:
# Does the intervention also move attention on the NEW data (which it never trained on)?
if NEW_ROOT:
    for tag, ck in [('newdata_baseline', CKPT), ('newdata_rrr_sky', 'artifacts/runs/rrr_sky/best.pt')]:
        !python interpret_quantify.py --config config_newdata_run.yaml --checkpoint "{ck}" \
            --sky-mask-dir artifacts/sky_masks_newdata --experiments auto --max-exp 5 \
            --num-per-exp 40 --target yaw --saliency inputgrad --tag {tag}
    for tag in ['newdata_baseline', 'newdata_rrr_sky']:
        d = json.load(open(f'artifacts/quantify/regions_{tag}.json'))
        print(tag, 'sky concentration %.2fx' % d['regions']['sky']['concentration']['mean'])

## Step 7 — λ sweep

Optional, and the slowest cell. Everything above uses λ=1; this checks the result is not a
knife-edge.

In [ ]:
SWEEP = [0.25, 0.5, 2.0]
for lam in SWEEP:
    name = f'sweep_l{lam}'
    subprocess.run(['python', 'finetune_rrr.py', '--config', CFG, '--checkpoint', CKPT,
                    '--sky-mask-dir', 'artifacts/sky_masks', '--mask-mode', 'sky',
                    '--lambda-rrr', str(lam), '--epochs', '3', '--target', 'yaw',
                    '--max-steps', '200', '--output', f'artifacts/runs/{name}/best.pt'], check=True)
    subprocess.run(['python', 'interpret_quantify.py', '--config', CFG,
                    '--checkpoint', f'artifacts/runs/{name}/best.pt',
                    '--sky-mask-dir', 'artifacts/sky_masks', '--experiments', 'auto',
                    '--max-exp', '6', '--num-per-exp', '40', '--target', 'yaw',
                    '--saliency', 'inputgrad', '--tag', name], check=True)

In [ ]:
shutil.make_archive('/kaggle/working/vit_motion_v3_artifacts', 'zip', 'artifacts')
print('done -> vit_motion_v3_artifacts.zip')